# CESLR — Continuous Ethiopian Sign Language Recognition (Kaggle training)

**Before running:**
1. Settings → Accelerator → **GPU T4 x2**, Internet **ON**
2. Attach your dataset (Add Input): it must contain `fullFrame-256x256px/{train,dev,test}/<fileid>/1/*.png`
   (run `preprocess/dataset_preprocess.py --process-image --multiprocessing` locally first to resize frames to 256×256)
3. The GitHub repo must contain the fixed `utils/decode.py` (pyctcdecode), corrected STM files and `fix_annotations.py` — cell 3 verifies this and stops if the repo is stale.

Expected: ~15–20 min/epoch on 2×T4 → full 70 epochs ≈ 18–23 h (resume across sessions is automatic, checkpoints save every epoch). 40 epochs is usually plenty.

In [ ]:
import os
if not os.path.exists('/kaggle/working/CESLR'):
    !git clone https://github.com/ethio-artifical/CESLR.git /kaggle/working/CESLR
%cd /kaggle/working/CESLR
!pip -q install pyctcdecode

In [ ]:
# Link the attached Kaggle dataset into the expected path
import glob, os
matches = glob.glob('/kaggle/input/**/fullFrame-256x256px', recursive=True)
assert matches, 'No fullFrame-256x256px folder found under /kaggle/input — attach your dataset.'
src = matches[0]
dst_parent = 'dataset/CESLR/CESLR-multisigner/features'
os.makedirs(dst_parent, exist_ok=True)
dst = os.path.join(dst_parent, 'fullFrame-256x256px')
if not os.path.exists(dst):
    os.symlink(src, dst)
print('linked:', dst, '->', src)
for split in ['train', 'dev', 'test']:
    print(split, 'sequences:', len(os.listdir(os.path.join(dst, split))))

In [ ]:
# Sanity checks: repo fixes present, data consistent, frames really 256x256
import numpy as np, cv2, glob, os

assert 'import ctcdecode' not in open('utils/decode.py').read(), \
    'utils/decode.py still uses linux-only ctcdecode — push the pyctcdecode version to GitHub.'

n = len(open('evaluation/slr_eval/CESLR-groundtruth-dev.stm', encoding='utf-8').readlines())
assert n == 150, f'stale dev STM in evaluation/slr_eval ({n} lines, expected 150) — push the fixed STMs.'

gd = np.load('preprocess/CESLR/gloss_dict.npy', allow_pickle=True).item()
print('glosses:', len(gd), '-> num_classes:', len(gd) + 1)

info = np.load('preprocess/CESLR/train_info.npy', allow_pickle=True).item()
sample = info[0]
frames = glob.glob('dataset/CESLR/CESLR-multisigner/features/fullFrame-256x256px/' + sample['folder'])
assert frames, f"no frames found for {sample['fileid']} — check dataset layout"
h, w = cv2.imread(frames[0]).shape[:2]
print(f"{sample['fileid']}: {len(frames)} frames at {w}x{h} (info says {sample['num_frames']})")
assert (w, h) == (256, 256), f'frames are {w}x{h}, not 256x256 — run dataset_preprocess.py --process-image first'
print('all sanity checks passed')

In [ ]:
# Train (auto-resumes from the newest checkpoint if one exists).
# Cross-session resume: /kaggle/working is wiped between sessions, so to continue
# a previous run, attach that notebook version's Output as an extra input —
# its checkpoints are picked up here automatically.
import glob, os
ckpts = sorted(glob.glob('work_dir/baseline_res18/*.pt')
               + glob.glob('/kaggle/input/**/dev_*_model.pt', recursive=True),
               key=os.path.getmtime)
resume = f'--load-checkpoints {ckpts[-1]}' if ckpts else ''
print('resuming from:', ckpts[-1] if ckpts else 'scratch')
!python main.py --config configs/baseline.yaml --device 0,1 {resume}

In [ ]:
# Final evaluation on the test split with the best dev checkpoint
import glob, re
best = min(glob.glob('work_dir/baseline_res18/dev_*_model.pt'),
           key=lambda p: float(re.search(r'dev_([\d.]+)_', p).group(1)), default=None)
assert best, 'no dev checkpoint found yet'
print('best checkpoint:', best)
!python main.py --config configs/baseline.yaml --device 0 --phase test --load-weights {best}